# 02 - Cells table

One row per cell, with what I need before building features: batch, charging policy, the cycle life stored in the file, and whether the cell actually reached end of life (0.88 Ah) in this batch.

The file helpers from notebook 01 now live in `src/load.py`.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.load import open_batch, n_cells, get_value, get_text, get_summary

EOL = 0.88  # 80% of the 1.1 Ah nominal capacity
# the summary stops one cycle before cycle_life, so the last stored value
# is just above 0.88 (about 0.8806). A small margin catches those cells.
EOL_CHECK = 0.885

Cell ids follow the `b1c0` pattern (batch 1, cell index 0), the same naming Severson et al. and BatteryML use, so I can match their train/test split later.

`QDischarge` at index 0 is skipped because the first cycle in batch 1 is empty (see notebook 01).

In [2]:
rows = []
for batch in [1, 2, 3]:
    f = open_batch(batch, root="../")
    for i in range(n_cells(f)):
        s = get_summary(f, i)
        q = s["QDischarge"][1:]
        rows.append({
            "cell_id": f"b{batch}c{i}",
            "batch": batch,
            "index": i,
            "policy": get_text(f, "policy_readable", i),
            "cycle_life_file": get_value(f, "cycle_life", i),
            "n_cycles": len(s["cycle"]),
            "q_start": q[0],
            "q_end": q[-1],
            "q_min": q.min(),
        })
    f.close()

cells = pd.DataFrame(rows)
cells["reached_eol"] = cells["q_min"] <= EOL_CHECK
cells.head()

,cell_id,batch,index,policy,cycle_life_file,n_cycles,q_start,q_end,q_min,reached_eol
0,b1c0,1,0,3.6C(80%)-3.6C,1190.0,1189,1.070689,1.026210,1.008352,False
1,b1c1,1,1,3.6C(80%)-3.6C,1179.0,1178,1.075301,1.038225,1.038225,False
2,b1c2,1,2,3.6C(80%)-3.6C,1177.0,1176,1.079922,1.043279,1.043279,False
3,b1c3,1,3,4C(80%)-4C,1226.0,1225,1.079723,0.970921,0.970921,False
4,b1c4,1,4,4C(80%)-4C,1227.0,1226,1.078417,1.021960,1.021641,False


In [3]:
print(len(cells), "cells")
cells.groupby("batch").agg(
    cells=("cell_id", "count"),
    reached_eol=("reached_eol", "sum"),
    life_min=("cycle_life_file", "min"),
    life_median=("cycle_life_file", "median"),
    life_max=("cycle_life_file", "max"),
)

140 cells


,cells,reached_eol,life_min,life_median,life_max
batch,,,,,
1,46,36,534.0,858.5,1227.0
2,48,48,148.0,483.0,1061.0
3,46,44,541.0,1005.5,1935.0


## Cells that did not reach end of life in their batch

In [4]:
cells.loc[~cells["reached_eol"], ["cell_id", "policy", "cycle_life_file", "n_cycles", "q_end"]]

,cell_id,policy,cycle_life_file,n_cycles,q_end
0,b1c0,3.6C(80%)-3.6C,1190.0,1189,1.026210
1,b1c1,3.6C(80%)-3.6C,1179.0,1178,1.038225
2,b1c2,3.6C(80%)-3.6C,1177.0,1176,1.043279
3,b1c3,4C(80%)-4C,1226.0,1225,0.970921
4,b1c4,4C(80%)-4C,1227.0,1226,1.021960
8,b1c8,5.4C(40%)-3.6C,879.0,878,0.969025
10,b1c10,5.4C(50%)-3C,906.0,905,0.961019
12,b1c12,5.4C(50%)-3.6C,902.0,901,0.913119
13,b1c13,5.4C(50%)-3.6C,897.0,896,0.923136
22,b1c22,6C(30%)-3.6C,891.0,890,0.963279


## Batch 2 cells that start with a used cell

A new cell starts around 1.07 Ah. The continued cells from batch 1 start lower and have a cut-off policy text.

In [5]:
b2 = cells[cells["batch"] == 2]
b2.loc[b2["q_start"] < 1.04, ["cell_id", "policy", "q_start", "n_cycles", "cycle_life_file"]]

,cell_id,policy,q_start,n_cycles,cycle_life_file
53,b2c7,80%)-3.6C,1.023337,662,663.0
54,b2c8,80%)-3.6C,1.026484,981,982.0
55,b2c9,80%)-3.6C,1.031750,1060,1061.0
61,b2c15,80%)-4C,0.971675,208,209.0
62,b2c16,80%)-4C,1.009642,482,483.0


Matching them by eye, using the end capacity in batch 1 and the start capacity in batch 2:

In [6]:
continued = {"b1c0": "b2c7", "b1c1": "b2c8", "b1c2": "b2c9", "b1c3": "b2c15", "b1c4": "b2c16"}
by_id = cells.set_index("cell_id")
pd.DataFrame([
    {
        "batch1_cell": a,
        "q_end_b1": by_id.loc[a, "q_end"],
        "batch2_cell": b,
        "q_start_b2": by_id.loc[b, "q_start"],
        "cycles_b1": by_id.loc[a, "n_cycles"],
        "cycles_b2": by_id.loc[b, "n_cycles"],
    }
    for a, b in continued.items()
]).round(3)

,batch1_cell,q_end_b1,batch2_cell,q_start_b2,cycles_b1,cycles_b2
0,b1c0,1.026,b2c7,1.023,1189,662
1,b1c1,1.038,b2c8,1.026,1178,981
2,b1c2,1.043,b2c9,1.032,1176,1060
3,b1c3,0.971,b2c15,0.972,1225,208
4,b1c4,1.022,b2c16,1.010,1226,482


## Save

`data/` is ignored by git, so the csv is not committed. Running this notebook rebuilds it.

In [7]:
cells.to_csv("../data/cells.csv", index=False)
print("saved", len(cells), "rows")

saved 140 rows


## Notes

- 140 rows. Batch 1 has 36 of 46 cells reaching 0.88 Ah, batch 2 has all 48 (some only because they are the continued batch 1 cells), and batch 3 has 44 of 46.
- The five continued cells line up well: the end capacity in batch 1 is within about 0.015 Ah of the start capacity in batch 2. For these five, the stored `cycle_life` in batch 1 is just where batch 1 stopped, not a real end of life. Their real life should be batch 1 cycles plus batch 2 cycles. I'll check that against the paper in step 1.5 before merging anything.
- `barcode` and `channel_id` are stored as MATLAB objects that h5py can't read as plain values, so I left them out. They're not needed for prediction.
- The table keeps every cell for now. Dropping and merging happens in step 1.5, where each decision gets written down.